# Importeren van libraries

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib as plt
from shared import init_analyse  #initiële analyse functie
#Importeren van libraries

# 1 ***Analyse sales data***

In dit onderdeel importeren en analyseren we de salesdata met behulp van pandas

### 1.1 Importeren en inlezen dataset

In [ ]:
sales_path= Path(r"Data\Contoso_Sales.csv")
#Inlezen sales.csv
sales_df = pd.read_csv(sales_path,
                       decimal=",")
#Dataframe koppelen aan variabele
#Decimal argument meegegeven, omdat pandas standaard "." als delimiter ziet.

Data is met behulp van pathlib en pandas ingelezen.  
Bij het inlezen ontstond initieel een dtypewarning voor de kolommen ReturnAmount en DiscountAmount. 
Door bij het inlezen het decimal teken mee te geven wordt deze fout verholpen en worden floats gelijk correct ingelezen.

### 1.2. Eerste analyse
Hier wordt de algehele dataset geanalyseerd op afwijkingen, waar hierna op groter detailniveau naar gekeken wordt.

In [ ]:
sales_df.info()
#Eerste data over kolommen opvragen

- Datekey wordt als string weergegeven, deze dient nog omgezet te worden naar een tijdreeks.
- UnitCost, UnitPrice, TotalCost en SalesAmount werden als strings geïnterpreteerd, door middel van decimal ingeven bij import gecorrigeeerd..

In [ ]:
sales_df.head(20)
#De eerste 20 rijn tonen met waarden.

Datekey lijkt een datumwaarde te moeten zijn, deze moet bij het opschonen van de data omgezet worden.  
UnitCost, UnitPrice, TotalCost en SalesAmount lijkt om bedragen te gaan. De stuksprijs maal de quantity is hier ook gelijk aan de Totalcost en SalesAmount.  
Gezien er geen strings in de tabel zitten ook geen noodzaak om te controleren of deze afwijken van elkaar in schrijfwijze.

In [ ]:
sales_df.tail(20)
#Laatste 20 waarden bekijken.

Laatste 20 rijen laten hetzelfde patroon zien. In deze rijen zitten ook returns verwerkt. Deze lijken wel goed verwerkt te worden in de TotalCost, bijvoorbeeld 10 items verkocht - 1 return = 9* unitcost.  
Op salesniveau lijkt dit echter niet correct doorberekend. De SalesAmount blijft gelijk aan 10 maal de UnitPrice, ondanks de return.  
Onderzoeken of dit elders wordt opgevangen. 

In [ ]:
sales_df.describe()

In [ ]:
sales_df.isna().sum()
#Controleren op lege waarden in cellen. Dit is niet van toepassing.
sales_df.nunique() 
#Aantal unique waarden opsommen.

In [ ]:
sales_df[sales_df.duplicated()]
#Checken op duplicated rows

Op basis van bovenstaande analyses zijn er geen lege waardes of duplicaten aanwezig in deze dataset.

De volgende keys dienen gecontroleerd te worden op aanwezigheid in de andere dataset:
- 4 verkoopkanalen volgens sales set.
- 306 unieke winkels volgens sales set.
- 1689 soort producten
- 28 promoties
- 783 retourregels

### 1.3 Corrigeren datatypes

Op basis van de analyse zien we dat de datum gegevens nu niet als juiste dtype worden ingegeven.  
In dit onderdeel worden deze omgezet naar bruikbaar datums.

In [ ]:
sales_df2 = sales_df.copy()
#Dataframe kopieeëren als backup 
sales_df2

In [ ]:
sales_df2["DateKey"] = pd.to_datetime(sales_df2["DateKey"], format="%Y-%m-%d %H:%M:%S,%f")
#Omzetten naar datetime van DateKey
display(sales_df2)

In [ ]:
print(sales_df2.loc[sales_df2["DateKey"].idxmin(),"DateKey"])
print(sales_df2.loc[sales_df2["DateKey"].idxmax(),"DateKey"])
#Kijken naar begin en einddatum van dataset.

De dataset loopt van 1 januari 2011 tot en met 31 december 2013

In [ ]:
datesales = sales_df2.set_index("DateKey")
datesales.sort_values(by="DateKey")

In [ ]:
datesales.resample("ME").agg(aantal_keys=("SalesKey","nunique"),
                             verkoopkanalen=("channelKey","nunique"),
                             kosten=("TotalCost","sum"),
                             SalesAmount=("SalesAmount","sum")
                             )

## **Voorlopige conclusie**

De sales dataset loopt van 1 januari 2011 tot en met 31 december 2013 en bestaat uit ***2.282.482*** unieke saleskeys.
Het is nog onduidelijk of de return amounts verrekend moeten worden met de SalesAmount, gezien het verschil tussen kosten en opbrengsten bij een retour.

De volgende keys dienen gecontroleerd te worden op aanwezigheid in de andere dataset:
- 4 **verkoopkanalen** volgens sales set.
- 306 **unieke winkels** volgens sales set.
- 1689 soort **producten**
- 28 **promoties**
- 783 **retourregels**

*Actiepunten*:  
- Navragen verrekenen returns.
- Afronden discountamount en salesamount.

# 2 ***Analyse product data***
In dit onderdeel wordt de product data set verkend.  
Het eerste doel is om een indruk te krijgen van de inhoud en de grote van de data set.  
Ten tweede wordt er gecontroleerd op eventuele afwijkende waarden, duplicaten, en missende waarden.  
Uiteindelijk wordt er ook gekeken naar de link met de sales data set. Naar verwachting zijn er minimaal 1689 producten in de dataset.

### 2.1 Importeren dataset
De dataset wordt geimporteerd met behulp van pandas en path.

In [ ]:
product_path= Path(r"Data\Contoso_Product.csv")
#Inlezen sales.csv
product_df = pd.read_csv(product_path,
                       decimal=",")

### 2.2 Analyseren product dataset

In [95]:
init_analyse(product_df)
#Functie aanhalen uit shared.py geschreven om te voorkoemn dat telkens dezelfde stappen doorlopen hoeven te worden.

informatie over dataframe:
<class 'pandas.DataFrame'>
RangeIndex: 1690 entries, 0 to 1689
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   BrandName              1690 non-null   str    
 1   ClassName              1690 non-null   str    
 2   Manufacturer           1690 non-null   str    
 3   ProductDescription     1690 non-null   str    
 4   ProductKey             1690 non-null   int64  
 5   ProductName            1690 non-null   str    
 6   ProductSubcategoryKey  1690 non-null   int64  
 7   UnitCost               1690 non-null   float64
 8   UnitPrice              1690 non-null   float64
dtypes: float64(2), int64(2), str(5)
memory usage: 119.0 KB

aantal gedupliceerde rijen: 0

Overzicht van lege waarden:


BrandName                0
ClassName                0
Manufacturer             0
ProductDescription       0
ProductKey               0
ProductName              0
ProductSubcategoryKey    0
UnitCost                 0
UnitPrice                0
dtype: int64


Eerste 20 rijen:


,BrandName,ClassName,Manufacturer,ProductDescription,ProductKey,ProductName,ProductSubcategoryKey,UnitCost,UnitPrice
0,A. Datum,Deluxe,A. Datum Corporation,"Digital camera, product type digital camera - ...",951,A. Datum Consumer Digital Camera E100 Black,23,143.26,281.0
1,A. Datum,Deluxe,A. Datum Corporation,"Digital camera, product type digital camera - ...",965,A. Datum Consumer Digital Camera E100 Grey,23,143.26,281.0
2,A. Datum,Deluxe,A. Datum Corporation,"Digital camera, product type digital camera - ...",979,A. Datum Consumer Digital Camera E100 Pink,23,143.26,281.0
3,A. Datum,Deluxe,A. Datum Corporation,"Digital camera, product type digital camera - ...",993,A. Datum Consumer Digital Camera E100 Silver,23,143.26,281.0
4,A. Datum,Deluxe,A. Datum Corporation,"Digital camera, product type digital camera - ...",1007,A. Datum Consumer Digital Camera E100 Orange,23,143.26,281.0
5,A. Datum,Deluxe,A. Datum Corporation,"Digital camera, product type digital camera - ...",1021,A. Datum Consumer Digital Camera E100 Green,23,143.26,281.0
6,A. Datum,Deluxe,A. Datum Corporation,"Digital camera, product type digital camera - ...",1035,A. Datum Consumer Digital Camera E100 Azure,23,143.26,281.0
7,A. Datum,Deluxe,A. Datum Corporation,"Digital camera, Product type digital camera - ...",950,A. Datum Super-zoom Digital Camera X300 Black,23,96.08,290.0
8,A. Datum,Deluxe,A. Datum Corporation,"Digital camera, Product type digital camera - ...",964,A. Datum Super-zoom Digital Camera X300 Grey,23,96.08,290.0
9,A. Datum,Deluxe,A. Datum Corporation,"Digital camera, Product type digital camera - ...",978,A. Datum Super-zoom Digital Camera X300 Pink,23,96.08,290.0



Laatste 20 rijen:


,BrandName,ClassName,Manufacturer,ProductDescription,ProductKey,ProductName,ProductSubcategoryKey,UnitCost,UnitPrice
1670,Wide World Importers,Regular,Wide World Importers,"High brightness, 1300 ANSI lumen light output,...",626,WWI Projector 480p LCD12 White,19,116.75,229.00
1671,Wide World Importers,Regular,Wide World Importers,"High brightness, 1300 ANSI lumen light output,...",637,WWI Projector 480p LCD12 Silver,19,116.75,229.00
1672,Wide World Importers,Regular,Wide World Importers,Listening to music wirelessly with other Bluet...,106,WWI Stereo Bluetooth Headphones New Generation...,6,61.16,132.99
1673,Wide World Importers,Regular,Wide World Importers,Listening to music wirelessly with other Bluet...,107,WWI Stereo Bluetooth Headphones New Generation...,6,61.16,132.99
1674,Wide World Importers,Regular,Wide World Importers,Listening to music wirelessly with other Bluet...,108,WWI Stereo Bluetooth Headphones New Generation...,6,61.16,132.99
1675,Wide World Importers,Regular,Wide World Importers,Listening to music wirelessly with other Bluet...,109,WWI Stereo Bluetooth Headphones New Generation...,6,61.16,132.99
1676,Wide World Importers,Regular,Wide World Importers,Listening to music wirelessly with other Bluet...,110,WWI Stereo Bluetooth Headphones New Generation...,6,61.16,132.99
1677,Wide World Importers,Regular,Wide World Importers,"Native 720p resolution, 1700 ANSI lumens, 2500...",614,WWI Projector 720p DLP56 Black,19,459.40,999.00
1678,Wide World Importers,Regular,Wide World Importers,"Native 720p resolution, 1700 ANSI lumens, 2500...",625,WWI Projector 720p DLP56 White,19,459.40,999.00
1679,Wide World Importers,Regular,Wide World Importers,"Native 720p resolution, 1700 ANSI lumens, 2500...",636,WWI Projector 720p DLP56 Silver,19,459.40,999.00


Uit de eerste analyse blijkt dat er 1690 unique keys in de dataset zitten. Dit sluit aan op de eerdere verwachting dat er minimaal 1689 unieke keys aanwezig dienen te zijn. Er is ook geen sprake van duplicaten op basis van deze analyse. Uit de weergave blijkt dat producten in verschillende kleuren beschikbaar zijn. Dit kan later interessant zijn voor een analyse op welk kleur product het beste/slechtste verkoopt. 

Gekeken naar de datatypes zou de verwachting zijn dat de UnitCost en Unitprice als floats zouden worden ingeladen. Dit lijkt voorkomen te worden door de dollartekens in de cellen. Dit dienen we nog op te schonen. 

In de hele product dataset komen geen lege waarden voor.

De eerste stap is om de kosten en prijs om te zetten naar decimale waarden. Hiermee kunnen we ook controleren of de kosten en prijs in de product dataset gelijk zijn aan die in de sales dataset.

In [ ]:
product_df["UnitCost"] = product_df["UnitCost"].str.removeprefix("$").str.replace(",",".")
product_df["UnitPrice"] = product_df["UnitPrice"].str.removeprefix("$").str.replace(",",".")
#Verwijderen van de $ prefixes uit de kolommen UnitCost en UnitPrice. Daarna de komma vervangen om floats te kunnen maken.

In [ ]:
product_df[["UnitCost","UnitPrice"]] = product_df[["UnitCost","UnitPrice"]].apply(pd.to_numeric)
#UnitCost en UnitPrice omzetten in floats.

In [ ]:
product_df.info()
#Controleren of waarden zijn omgezet.

In [117]:
product_df.loc[~product_df["ProductKey"].isin(sales_df["ProductKey"])]
#Zoeken naar producten wel in product dataset maar niet in de salesdata set

,BrandName,ClassName,Manufacturer,ProductDescription,ProductKey,ProductName,ProductSubcategoryKey,UnitCost,UnitPrice
35,A. Datum,Economy,A. Datum Corporation,"Digital camera, 3.7 in x 0.9 in x 2",944,A. Datum Compact Digital Camera M200 Black,23,59.32,129.0


De A. Datum Comapct Digital Camera M200 black deze zit wel in de product dataset maar wordt volgens de sales data niet verkocht.

In [118]:
display(product_df.nunique())
#Aantal unieke eigenschappen onderzoeken.

BrandName                  10
ClassName                   3
Manufacturer               10
ProductDescription        516
ProductKey               1690
ProductName              1690
ProductSubcategoryKey      22
UnitCost                  397
UnitPrice                 350
dtype: int64

In [ ]:
#Onderzoeken van strings op verschillende soorten en eventuele dubbel aanwezige gegevens met afwijkende schrijfwijzes.

display(product_df["BrandName"].value_counts())
#Onderzoeken producten van ieder merk
display(product_df["ClassName"].value_counts())
#Onderzoeken segmenten
display(product_df["Manufacturer"].value_counts())
#Onderzoeken Producenten
display(product_df["ProductDescription"].value_counts())
#Onderzoeken productomschrijvingen
display(product_df["ProductName"].value_counts())
#Onderzoeken productnamen

De verschillende merken, segmenten, producenten, productomschrijvingen en productnamen zijn geanalyseerd.
Hier komen geen dubbele waarde in voor veroorzaakt door afwijkende schrijfwijzen.   

Opvallend is het feit dat er **516** unieke descripties zijn van producten, maar **1690** unieke productbenamingen.
Dit wordt met name veroorzaakt door het feit dat iedere product meerdere kleurvarianten heeft.   

Het lijkt aan te raden om een scheiding te maken tussen productnamen en kleuren. 
Zo kan er zuiver gekeken worden naar productverkopen en later naar de voorkeur van de kleur. 
De verwachting is dat na opschonen het aantal unieke productnamen zonder kleur gelijk is aan de unieke productomschrijvingen.

De volgende stap is om te kijken of product kosten en prijzen overeenkomen met de gegevens uit de sales set.

In [ ]:
sales_df["ProductUnitCost"] = sales_df["ProductKey"].map(
    product_df.set_index("ProductKey")["UnitCost"]
)

sales_df["CostControl"] = (
    sales_df["ProductUnitCost"] == sales_df["UnitCost"]
)

In [ ]:
sales_df[~sales_df["CostControl"]]
#Controleren waar UnitCost niet overeenkomt.

In [ ]:
sales_df["ProductUnitPrice"] = sales_df["ProductKey"].map(
    product_df.set_index("ProductKey")["UnitPrice"]
)

sales_df["PriceControl"] = (
    sales_df["ProductUnitPrice"] == sales_df["UnitPrice"]
)

In [ ]:
sales_df[~sales_df["PriceControl"]]
#Onderzoeken waar prijs niet overeenkomt.

Uit vergelijkingen met de sales dataset blijkt dat kosten en prijzen voor alle sales overeenkomen met die uit de productdata.

## **Voorlopige conclusie**

Conform eerdere verwachtingen bevat de product dataset minimaal 1689 productkeys, het totaal is 1690. 
Het enige product dat niet voorkomt in de sales dataset is de *A. Datum Comapct Digital Camera M200 black*.  
Verder is er gekeken naar volgende eigenschappen:
- Merknamen - 10 in totaal
- Segmenten - 3 in totaal
- Producten -10 in totaal
- Unieke productomschrijvingen - 516 in totaal
- Unieke productnamen - 1690 in totaal.

Er is in deze benamingen geen besprake van duplicaten ontstaan door bijvoorbeeld verschillende schrijfwijzes.  
Het is wel opvallend dat er 516 productomschrijvingen zijn, tegenover 1690 productnamen. Nadere analyse toont aan dat dit wordt veroorzaakt door het feit dat producten verkrijgbaar zijn in meerdere kleuren. 

*Actiepunten*:
- Aparte kolommen toevoegen voor productnaam en kleur.

# 3 ***Analyse calendar data***
In dit onderdeel kijken we naar de calendar dataset.
Naar verwachting zal deze matchen op de datekey en minimaal datums van 1 januari 2011 tot en met 31 december 2013 bevatten.

### 3.1 Importeren dataset
Met behulp van path en pandas importeren wordt de dataframe geïmporteerd voor analyse.

In [ ]:
calendar_path= Path(r"Data\Contoso_calendar.csv")
#Inlezen calendar.csv als raw string om speciale tekens te escapen.
calendar_df = pd.read_csv(calendar_path,
                       decimal=",")

### 3.2 Analyse dataset
Hier voeren we de eerste analyse uit op de calendar dataset.

In [ ]:
init_analyse(calendar_df)
#Met behulp van analyse functie eerste algemene gegeens opvragen.

De dataset is groter dan de sales dataset qua datumbereik. Dit is opzich geen probleem, echter kan dit wel voor extra load zorgen gezien gegevens die niet worden gebruikt worden ingeladen.
Een actiepunt is om eventueel later te kijken naar het schrappen van datums die niet relevant zijn om performance te verbeteren.
Alle waarden zijn gevuld en volgen een standaard structuur. 
De Year kolom lijkt 2 keer aanwezig, eventueel kan er één gedropt worden uit de set. Daarnaast lijken Day en DayOfMonth dezelfde gegevens te weergeven. 

Op het moment is het niet zeker of deze tabel in PowerBI direct werkt, voor pandas gebruik hebben we een kolom nodig die als datetime functioneert. 
De volgende stap is om op voorhand een kolom te maken die als datetime gebruikt kan worden. 

In [ ]:
calendar_df["DateInt"].isin(sales_df["DateKey"])
#Controleren matches in sales dataset.

In de sales data wordt gebruik gemaakt van een DateKey kolom. Deze waarde komt niet terug in de calendar dataset en matcht ook niet op DateInt.
Volgende stap is  het aanmaken van een key die matcht op de key in de sales set om datums te verbinden.

### 3.3 Datetime value maken
In dit onderdeel maken we met de aanwezige gegevens een DateKey om te matchen op de waarde in de sales set.

In [ ]:
calendar_df.head(20)

In [ ]:
calendar_df["DateKey"] = pd.to_datetime(calendar_df["DateInt"], format="%Y%m%d")
#Datekey aanmaken in calendar set door de DateInt te converteren naar een datetime op basis van het format.


In [ ]:
calendar_df[calendar_df["DateKey"].isin(sales_df2["DateKey"])]
#Controleren of de DateKey matcht op waarden in de sales data. 

Met behulp van de DateIn waarde is een DateKey aangemaakt die matcht op gegevens in de sales dataset. 
Bij Controle van de dataset start deze op 1 januari 2011 en eindigt deze op 31 december 2013. Dit komt overeen met de sales dataset.
Dit lijkt voornamelijk relevant voor het PowerBI gedeelte.

## **Voorlopige conclusie**

De calendar data lijkt met name voor de tijdfunctie binnen PowerBI.
Voor de zekerheid is een key aangemaakt om op te matchen met de sales data.  
Day en DayOfMonth lijken dezelfde waarde te bevatten, idem voor Year en Year.1

*actiepunten*:
- Eventueel dubbele waarden voor day en year laten vallen.

# 4 ***Analyse geography data***
In dit onderdeel wordt de product dataset verkend.  
Het eerste doel is om een indruk te krijgen van de inhoud en de grote van de data set.  
Ten tweede wordt er gecontroleerd op eventuele afwijkende waarden, duplicaten, en missende waarden.  
Uiteindelijk wordt er ook gekeken naar de link met de sales data set. 

### 4.1 Importeren dataset


In [ ]:
geography_path= Path(r"Data\Contoso_Geography.csv")
#Inlezen geography.csv
geography_df = pd.read_csv(geography_path,
                       decimal=",")

### 4.2 Analyse dataset

In [ ]:
init_analyse(geography_df)

De dataset lijkt opgebouwd uit contintenten, landen, provincies/regios en steden.  
Bij regioncountryname zijn er 3 lege waarden gevonden in de initiële analyse. Hier dient op ingezoomd te worden. Er is geen sprake van duplicated.

In [ ]:
geography_df[geography_df["RegionCountryName"].isna()]
#Waarden met ontbrekende regioncountryname bekijken.

De lege waarden lijken betrekking hebben op het hoogste niveau, contintent. Dit lijkt logisch, eventueel kan hier later een mock waarde ingegeven worden. 

In [ ]:
geographystr = ["ContinentName","GeographyType","RegionCountryName"]

for g in geographystr:
    display(geography_df[g].value_counts())
#Onderzoeken dataset op afwijkende schrijfwijzes, daarnaast de samenstelling van de geography set.

## Voorlopige conclusie
De geography dataset bevate gegevens omtrent locaties. Deze kunnen niet direct gelinkt worden aan de sales dataset.  
De voorlopige aanname is dat deze samenhangt met store data.
De gegevens hebben met name betrekking op de Verenigde Staten, daarop volgt Europa en uiteindelijk Europa.

Het zou interessant kunnen zijn om ontwikkelingen van de verkopen op deze verschillende contintenten te zien en vergelijken. 
Eventueel product context.

*actiepunten*:
- Controleren of er samenhang is met store dataset.

# 5 ***Analyse store data***
In dit onderdeel wordt de product dataset verkend.  
Het eerste doel is om een indruk te krijgen van de inhoud en de grote van de data set.  
Ten tweede wordt er gecontroleerd op eventuele afwijkende waarden, duplicaten, en missende waarden.  
Uiteindelijk wordt er ook gekeken naar de link met de sales dataset en de geography dataset

### 5.1 Data importeren

In [ ]:
stores_path= Path(r"Data\Contoso_Stores.csv")
#Inlezen stores.csv
stores_df = pd.read_csv(stores_path,
                       decimal=",")

### 5.2 Data analyseren

In [ ]:
init_analyse(stores_df)

Volgens de dataset zijn er 306 unieke storekeys dit komt overeen met eerdere analyse uit de sales data.
Met name in de CloseReason kolom zijn veel lege waarden. De aanname is dat deze pas wordt ingevuld bij een sluiting van een locatie.
Er is geen data aanwezig over wanneer de winkel is gesloten, het lijkt dus niet mogelijk te oordelen of er foutieve toewijzigingen aan een winkel zijn in de salessset.
Storetypes lijkt samen te kunnen hangen met de verkoopkanalen die voort komen uit de sales dataset.

Daarnaast ontbreekt er bij 1 store een employeecount, hier dient naar gekeken te worden. Tevens worden employeecounts op het moment als floats opgeslagen, ervanuit gaande dat dit om personen gaat en niet FTE.

Zoals eerder verwacht is de geographykey verwerkt in deze dataset.

In [ ]:
Control = ["CloseReason","Status","StoreName","StoreType"]
display(stores_df.nunique())

for c in Control:
    display(stores_df[c].value_counts())
#Loop om values te weergeven bij unieke waardes.



Volgende feiten vallen op gekeken naar de verschillende omschrijvingen en namen:
- 2 sluitingsreden
- 263 geographykeys t.o.v. 674 in de geography set
- 15 selling area sizes, de vraag is of dit daadwerkelijke oppervlakte is of een soort bucket?
- 4 storetypes, lijkt dus gelijk te zijn aan de verkoopkanalen.
 

In [ ]:
stores_df["StoreKey"].isin(sales_df2["StoreKey"])
sales_df2["StoreKey"].isin(stores_df["StoreKey"])
#Checken of alle waarden aanwezig zijn in salesset

In [ ]:
geography_df[~geography_df["GeographyKey"].isin(stores_df["GeographyKey"])]
#Waarden niet in store set.

geography_df[geography_df["GeographyKey"].isin(stores_df["GeographyKey"])]
#Waarden in de store set.

## Voorlopige Conclusie
Storekeys in de salesset komen overeen met de sales set. 
Een groot aantal geography locaties komen niet voor in de storeset.
Dit betekent niet dat ze niet relevant zijn, hier kunnen verkopen zijn via andere kanalen. 
Dit analyseren aan de hand van de channel dataset.

Wellicht interessant om in kaart te brengen of er locaties/regio's zijn waar veel verkopen zijn via andere kanalen waar een store van toegevoegde waarde zou zijn.

Er zijn geen historische gegevens beschikbaar tot en met de relocates. Daarnaast is de CloseReason "Store" niet geheel duidelijk.

# 6 ***Analyse Promotions data***
In dit onderdeel wordt de promotions dataset verkend.  
Het eerste doel is om een indruk te krijgen van de inhoud en de grote van de data set.  
Ten tweede wordt er gecontroleerd op eventuele afwijkende waarden, duplicaten, en missende waarden.  
Uiteindelijk wordt er ook gekeken naar de link met de sales data.

### 6.1 Data importeren

In [ ]:
promotions_path= Path(r"Data\Contoso_Promotion.csv")
#Inlezen promotions.csv
promotions_df = pd.read_csv(promotions_path)
#Decimal weggehaald, gezien set met . werkt voor decimalen.


### 6.2 Data analyse

In [88]:
init_analyse(promotions_df)

informatie over dataframe:
<class 'pandas.DataFrame'>
RangeIndex: 28 entries, 0 to 27
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   DiscountPercent  28 non-null     float64
 1   Year             28 non-null     int64  
 2   Quarter          28 non-null     str    
 3   Month            28 non-null     str    
 4   Day              28 non-null     int64  
 5   PromotionKey     28 non-null     int64  
 6   PromotionLabel   28 non-null     int64  
 7   PromotionName    28 non-null     str    
 8   Year.1           28 non-null     int64  
 9   Quarter.1        28 non-null     str    
 10  Month.1          28 non-null     str    
 11  Day.1            28 non-null     int64  
dtypes: float64(1), int64(6), str(5)
memory usage: 2.8 KB

aantal gedupliceerde rijen: 0

Overzicht van lege waarden:


DiscountPercent    0
Year               0
Quarter            0
Month              0
Day                0
PromotionKey       0
PromotionLabel     0
PromotionName      0
Year.1             0
Quarter.1          0
Month.1            0
Day.1              0
dtype: int64


Eerste 20 rijen:


,DiscountPercent,Year,Quarter,Month,Day,PromotionKey,PromotionLabel,PromotionName,Year.1,Quarter.1,Month.1,Day.1
0,0.05,2011,Qtr 1,March,31,2,2,North America Spring Promotion,2011,Qtr 1,January,1
1,0.20,2011,Qtr 2,April,30,6,6,Asian Spring Promotion,2011,Qtr 1,February,1
2,0.07,2011,Qtr 2,April,30,8,8,European Spring Promotion,2011,Qtr 1,February,1
3,0.10,2011,Qtr 2,June,30,7,7,Asian Summer Promotion,2011,Qtr 2,May,1
4,0.10,2011,Qtr 3,September,30,3,3,North America Back-to-School Promotion,2011,Qtr 3,July,1
5,0.10,2011,Qtr 3,September,30,9,9,European Back-to-Scholl Promotion,2011,Qtr 3,August,1
6,0.20,2011,Qtr 4,December,31,4,4,North America Holiday Promotion,2011,Qtr 4,November,1
7,0.15,2012,Qtr 1,January,31,5,5,Asian Holiday Promotion,2011,Qtr 4,November,1
8,0.20,2012,Qtr 1,January,31,10,10,European Holiday Promotion,2011,Qtr 4,October,1
9,0.05,2012,Qtr 1,March,31,11,11,North America Spring Promotion,2012,Qtr 1,January,1



Laatste 20 rijen:


,DiscountPercent,Year,Quarter,Month,Day,PromotionKey,PromotionLabel,PromotionName,Year.1,Quarter.1,Month.1,Day.1
8,0.20,2012,Qtr 1,January,31,10,10,European Holiday Promotion,2011,Qtr 4,October,1
9,0.05,2012,Qtr 1,March,31,11,11,North America Spring Promotion,2012,Qtr 1,January,1
10,0.20,2012,Qtr 2,April,30,15,15,Asian Spring Promotion,2012,Qtr 1,February,1
11,0.07,2012,Qtr 2,April,30,17,17,European Spring Promotion,2012,Qtr 1,February,1
12,0.10,2012,Qtr 2,June,30,16,16,Asian Summer Promotion,2012,Qtr 2,May,1
13,0.10,2012,Qtr 3,September,30,12,12,North America Back-to-School Promotion,2012,Qtr 3,July,1
14,0.10,2012,Qtr 3,September,30,18,18,European Back-to-Scholl Promotion,2012,Qtr 3,August,1
15,0.20,2012,Qtr 4,December,31,13,13,North America Holiday Promotion,2012,Qtr 4,November,1
16,0.15,2013,Qtr 1,January,31,14,14,Asian Holiday Promotion,2012,Qtr 4,November,1
17,0.20,2013,Qtr 1,January,31,19,19,European Holiday Promotion,2012,Qtr 4,October,1


DiscountPercent is in tegenstelling tot voorgaande datasets met een . voor decimalen aangegeven. 
Er zijn 2 datum waarden aanwezig in de set. Een eerste analyse lijkt er op te wijzen dat dit een begin en eind datum van de promotie is, waarbij de datums eindigend op .1 de startdatums zijn. 
Door deze om te zetten naar datums kunnen we dit later gebruiken om te analyseren hoeveel er gebruik gemaakt is van de promoties in de periode.
Eventueel vergelijkingen maken tussen periodes met en zonder promotie en de winstgevendheid hiervan.  
Het aantal promotionkeys komt overeen met het aantal in de sales set. Promotionlabel lijkt altijd gelijk te zijn aan de key, onduidelijk waarom.  
Er zitten ook waarden in de dataset met betrekking tot 2014, deze kunnen niet gerelateerd worden aan de salesdata gezien deze loopt tot 31 december 2023.
Promotion names lijken grotendeels per jaar overeen te komen. Effectiviteit over de afgelopen 3 jaar zou dus bekeken kunnen worden. 
Eventueel waarde "European Back-to-Scholl Promotion" opschonen naar School.

Key 1 lijkt een doorlopende promotie te zijn om verkopen met geen korting vast te leggen.
Is er correlatie tussen promotion en discountamount? 
De volgende stap is om de samenhang met de salesdata te analyseren. Er is hier een aansluiting op de SalesKey.



In [ ]:
display(promotions_df["PromotionKey"].isin(datesales["PromotionKey"]).value_counts())
display(datesales["PromotionKey"].isin(promotions_df["PromotionKey"]).value_counts())
#Controleren welke promotionkeys in de salesset zitten en vice versa.


In [ ]:
datesales.loc[datesales["PromotionKey"] == 1, "DiscountAmount"].sum()
#Controleren of alle gegevens met promotionkey 1 geen discoutn hebben. 

In [ ]:
promo2 = datesales.loc[datesales["PromotionKey"]==2].sort_index()
#Alle waarden voor promo 2 opzoeken en sorteren op index en in aparte dataframe zetten.

In [ ]:
#Controleren discountamount op basis van promotionkey 2.
#Verwachting DiscountPct is 5% van aanschafwaarde
promo2["SalesBeforeDiscount"] = promo2["UnitPrice"]*promo2["SalesQuantity"]
promo2["DiscountPct"] = promo2["DiscountAmount"]/promo2["SalesBeforeDiscount"]*100
#Geen directe aansluiting tussen percentage en discountamount.
#Vermoeden UnitPrice*DiscountQuantity* DiscountPercent = DiscountAmount
promo2["DiscountPercent"] = promo2["PromotionKey"].map(promotions_df.set_index("PromotionKey")["DiscountPercent"])
#Discountpercentage koppelen aan waarde in promo dataframe. Zoek promotion key in promo en match deze op promotion dataset, voegt verolgens de discountpercent toe als kolom.
promo2["DiscountControle"] = promo2["UnitPrice"]*promo2["DiscountQuantity"]*promo2["DiscountPercent"]
#Discount herrekenen ter controle
promo2a = promo2.loc[(promo2["DiscountControle"].round(2) != promo2["DiscountAmount"].round(2)),["DiscountAmount","DiscountControle"]]
promo2a["ControleDiscountAmount"] = promo2a["DiscountAmount"] - promo2a["DiscountControle"]
promo2a.head(50)
promo2a[(promo2a["ControleDiscountAmount"]<-0.001)|(promo2a["ControleDiscountAmount"]>0.001)]
#Controleren afwijkingen, lijkt alleen op afrondingen terecht te komen.

Op basis van PromotionKey 2 is het verband tussen de PromotionKey het discountpercentage in de promotions dataset en de discountamount in de sales data set geanalyseerd.
De UnitPrice * DiscountQuantity * DiscountPercent is gelijk aan DiscountAmount voor deze promocode.
Op basis hiervan lijken er specifieke voorwaarden aan promotieacties te zitten over de hoeveelheden waarop korting wordt geleverd.
De volgende actie is om deze controle voor alle promotioncodes uit te voeren in de salesdataset.

In [ ]:
promocontrole = datesales.copy()
#Kopieeren van salesdataset
promocontrole["DiscountPercent"] = promocontrole["PromotionKey"].map(promotions_df.set_index("PromotionKey")["DiscountPercent"])
#Mappen van discountpercentage
promocontrole["DiscountControle"] = promocontrole["UnitPrice"]*promocontrole["DiscountQuantity"]*promocontrole["DiscountPercent"]
#Berekeken discountamount
promocontrole["ControleDiscountAmount"] = promocontrole["DiscountAmount"]-promocontrole["DiscountControle"]
display(promocontrole[promocontrole["ControleDiscountAmount"] != 0])
#Controleren waarden niet gelijk aan 0
display(promocontrole[(promocontrole["ControleDiscountAmount"]<-0.001)|(promocontrole["ControleDiscountAmount"]>0.001)])
#Controleren afrondingsverschillen.

Nu testen we dit voor de gehele dataset.

In [ ]:
datesales["DiscountPercent"] = datesales["PromotionKey"].map(promotions_df.set_index("PromotionKey")["DiscountPercent"])
datesales["DiscountControle"] = datesales["UnitPrice"]*datesales["DiscountQuantity"]*datesales["DiscountPercent"]
datesales["ControleDiscountAmount"] = datesales["DiscountAmount"]-datesales["DiscountControle"]
#Herhalen stappen voor controle

display(datesales[datesales["ControleDiscountAmount"] != 0])
#Dataframe weergeven waar controle waarde niet gelijk is aan 0.
display(datesales[(datesales["ControleDiscountAmount"]<-0.001)|(datesales["ControleDiscountAmount"]>0.001)])
#Dataframe met waardes met een afwijking groter dan een duizendste achter de decimaal.

Bovenstaande controle bevestigt het vermoeden dat UnitPrice * DiscountQuantity * DiscountPercent gelijk is aan DiscountAmount.
Er lijkt dus een verband te zijn tussen promotion en de hoeveelheid items. 
Promotions kunnen bijvoorbeeld zijn vanaf een bepaalde hoeveelheid of bij een bepaalde hoeveelheid een product gratis. Het is niet helemaal duidelijk voor iedere promotie wat deze verbande zijn en of deze relevant zijn.

In [ ]:
datesales.loc[(datesales["PromotionKey"]==2)&(datesales["DiscountAmount"]>0)].sort_index().head(50)
#Kijken naar eerste rijen met DiscountQuantity en PromotionKey 2.
datesales.loc[(datesales["PromotionKey"]==2)&(datesales["DiscountAmount"]>0)].sort_index().tail(50)
#Laate 50 waarden


In [ ]:
datesales.loc[datesales["PromotionKey"]==2].groupby("SalesQuantity")["DiscountQuantity"].describe()
#Kijken naar ratio's verkoop hoeveelheid afgezet tegen hoeveelheid discounts.

Voor nu lijkt er verder geen logisch verband te zijn tussen promotion acties en wanneer deze wel of niet wordt gegeven en op welke hoeveelheid producten.

## Voorlopige Conclusie
De promotieset bevat data over 28 promoties die open vanaf 1 januari 2011 tot en met 31 maart 2014.
De Promotion met Key 1 lijkt de standaardwaarde te zijn voor verkopen zonder enige korting.
Het verband met de salesset is het feit dat UnitPrice * DiscountQuantity * DiscountPercent gelijk is aan de DiscountAmount.

Er is geen duidelijk verband te vinden tussen de verschillende promotieacties en hoeveelheden. De verwachting zou zijn dat kortingen gelden vanaf een aantal items of per aantal items worden toegepast. Dit verband is echter niet te herleiden.

**Actiepunten**:
- De set heeft 2 datumwaarden. De eerste heeft betrekking op de einddatum de tweede op de begindatum van de promotion. Deze dienen nog duidelijk benoemd te worden. 
- Er zit een fout in de omschrijving van de Europese back-to-scholl actie. Deze nog aanpassen naar "European Back-to-School Promotion".

# 7 ***Analyse Product Category***

In [89]:
productcat_path= Path(r"Data\Contoso_Product_Category.csv")
#Inlezen product categories.csv
productcat_df = pd.read_csv(productcat_path)
#Decimal weggehaald, gezien set met . werkt voor decimalen.

In [93]:
init_analyse(productcat_df)

informatie over dataframe:
<class 'pandas.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 2 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   ProductCategory     8 non-null      str  
 1   ProductCategoryKey  8 non-null      int64
dtypes: int64(1), str(1)
memory usage: 260.0 bytes

aantal gedupliceerde rijen: 0

Overzicht van lege waarden:


ProductCategory       0
ProductCategoryKey    0
dtype: int64


Eerste 20 rijen:


,ProductCategory,ProductCategoryKey
0,Audio,1
1,Cameras and camcorders,4
2,Cell phones,5
3,Computers,3
4,Games and Toys,7
5,Home Appliances,8
6,"Music, Movies and Audio Books",6
7,TV and Video,2



Laatste 20 rijen:


,ProductCategory,ProductCategoryKey
0,Audio,1
1,Cameras and camcorders,4
2,Cell phones,5
3,Computers,3
4,Games and Toys,7
5,Home Appliances,8
6,"Music, Movies and Audio Books",6
7,TV and Video,2


Dataset lijkt een subcategory te zijn van een andere dataset. 
Vermoedelijk de Product Sub Category dataset.
Verbinding lijkt te zijn ProductKey -> ProductSubCategroy -> ProductCategory
Eerst product Sub Category bekijken.

# 8 ***Analyse Product Sub Category***

In [98]:
productsubcat_path= Path(r"Data\Contoso_ProductSubCategory.csv")
#Inlezen product subcategories.csv
productsubcat_df = pd.read_csv(productsubcat_path)
#Decimal weggehaald, gezien set met . werkt voor decimalen.

In [99]:
init_analyse(productsubcat_df)

informatie over dataframe:
<class 'pandas.DataFrame'>
RangeIndex: 44 entries, 0 to 43
Data columns (total 3 columns):
 #   Column                 Non-Null Count  Dtype
---  ------                 --------------  -----
 0   ProductCategoryKey     44 non-null     int64
 1   ProductSubcategory     44 non-null     str  
 2   ProductSubcategoryKey  44 non-null     int64
dtypes: int64(2), str(1)
memory usage: 1.2 KB

aantal gedupliceerde rijen: 0

Overzicht van lege waarden:


ProductCategoryKey       0
ProductSubcategory       0
ProductSubcategoryKey    0
dtype: int64


Eerste 20 rijen:


,ProductCategoryKey,ProductSubcategory,ProductSubcategoryKey
0,1,Audio Accessories,8
1,1,Bluetooth Headphones,6
2,1,Headphones,5
3,1,MP4&MP3,1
4,1,Radio,3
5,1,Recorder,2
6,1,Recording Pen,4
7,1,Speakers,7
8,2,Car Video,13
9,2,Home Theater System,11



Laatste 20 rijen:


,ProductCategoryKey,ProductSubcategory,ProductSubcategoryKey
24,4,Digital SLR Cameras,24
25,4,Film Cameras,25
26,5,Cell phones Accessories,33
27,5,Home & Office Phones,29
28,5,Smart phones & PDAs,32
29,5,Touch Screen Phones,31
30,6,Audio Books,36
31,6,Movie DVD,35
32,6,Music CD,34
33,7,Boxed Games,38


Er lijken 44 unieke Subcategorieën te zijn, verbonden aan 1 van de 8 hoofdcategorieën.
Volgende stap is om te kijken hoeveel van de subcategorykeys zich in de productlijst bevinden.

De header ProductSubcategory is niet in consistente camelcase

In [116]:
masker = productsubcat_df["ProductSubcategoryKey"].isin(product_df["ProductSubcategoryKey"])
#filter die controleert welke keys wel of niet in de product dataset zitten.
print(f"{productsubcat_df.loc[masker,["ProductSubcategoryKey"]].nunique().sum()} Productsubcategorieën in product dataframe")
display(productsubcat_df[masker])
#Geeft alle waarden weer die wel in de dataset zitten.
print(f"{productsubcat_df.loc[~masker,["ProductSubcategoryKey"]].nunique().sum()} Productsubcategorieën niet in product dataframe")
display(productsubcat_df[~masker])

22 Productsubcategorieën in product dataframe


,ProductCategoryKey,ProductSubcategory,ProductSubcategoryKey
1,1,Bluetooth Headphones,6
3,1,MP4&MP3,1
6,1,Recording Pen,4
8,2,Car Video,13
9,2,Home Theater System,11
10,2,Televisions,9
12,2,VCD & DVD,10
14,3,Computers Accessories,22
15,3,Desktops,17
16,3,Laptops,15


22 Productsubcategorieën niet in product dataframe


,ProductCategoryKey,ProductSubcategory,ProductSubcategoryKey
0,1,Audio Accessories,8
2,1,Headphones,5
4,1,Radio,3
5,1,Recorder,2
7,1,Speakers,7
11,2,TV & Video Accessories,14
13,3,Computer Setup & Service,21
18,3,Netbooks,16
25,4,Film Cameras,25
30,6,Audio Books,36


Dataset lijkt met namen hardware en multimedia apparaten te betreffen.  De 22 waarden sluiten ook aan met de unique count van de productset.

De productgroep die niet in de productcategory voorkomt heet met name berekking op witgoed, media en audio.

In [ ]:
productsubcat_df["ProductCategoryKey"].isin(productcat_df["ProductCategoryKey"])
productcat_df["ProductCategoryKey"].isin(productsubcat_df["ProductCategoryKey"])
#Controleren of alle keys in de productcategoryKey zitten.

0    True
1    True
2    True
3    True
4    True
5    True
6    True
7    True
Name: ProductCategoryKey, dtype: bool

In [135]:
print("Categorieën in product data set:")
display(productcat_df[(productcat_df["ProductCategoryKey"]).isin(productsubcat_df.loc[masker,"ProductCategoryKey"])])
#Controleren welke productcategorieën voorkomen in products
print("Categorieën niet in product data set:")
productcat_df[~(productcat_df["ProductCategoryKey"]).isin(productsubcat_df.loc[masker,"ProductCategoryKey"])]
#Controleren welke productcategorieën niet voorkomen in products

Categorieën in product data set:


,ProductCategory,ProductCategoryKey
0,Audio,1
1,Cameras and camcorders,4
2,Cell phones,5
3,Computers,3
6,"Music, Movies and Audio Books",6
7,TV and Video,2


Categorieën niet in product data set:


,ProductCategory,ProductCategoryKey
4,Games and Toys,7
5,Home Appliances,8


Conform de hypothese zijn witgoed en media uitgesloten van de set.

## Voorlopige conclusie

De dataset lijkt 22 van de 44 subcategorieën te gebruiken.
Daarnaast zijn de categoriën met ProductCategoryKey 7 en 8 niet van toepassing op de sales dataset.

**Actiepunten**:
- Eventueel ongebruikte categorieën laten vallen, gezien de hoeveelheid lijkt de impact hiervan niet al te groot op performance.

# 9 ***Analyse Channel data***

In [136]:
channel_path= Path(r"Data\Contoso_Channel.csv")
#Inlezen product channel.csv
channel_df = pd.read_csv(channel_path)
#Decimal weggehaald, gezien set met . werkt voor decimalen.

In [137]:
channel_df

,Channel,ChannelName
0,1,Store
1,2,Online
2,3,Catalog
3,4,Reseller


Er zijn 4 channels, deze corresponderen met de channelkeys in de sales set.